In [1]:
# ============================================================
# CELL 1 -- VERIFY OPENAI CREDENTIALS (no calls, no tokens)
# ============================================================
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

# -- Validate key format before doing anything -----------------
assert OPENAI_API_KEY, "OPENAI_API_KEY is empty. Set it in .env"
assert OPENAI_API_KEY.startswith("sk-"), f"Key looks wrong (starts with '{OPENAI_API_KEY[:5]}...')"

# -- Test connection: list models (free, no tokens used) -------
client = OpenAI(api_key=OPENAI_API_KEY)

try:
    models = client.models.list()
    available = [m.id for m in models.data if MODEL in m.id]
    print(f"[OK] Connected to OpenAI")
    print(f"[OK] API key: {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}")
    if available:
        print(f"[OK] Model '{MODEL}' available")
    else:
        print(f"[WARN] '{MODEL}' not found in your models. You might lack access.")
        print(f"       Available gpt models: {[m.id for m in models.data if 'gpt' in m.id][:5]}")
    print(f"\n  Ready. No tokens used.")
except Exception as e:
    print(f"[FAIL] {type(e).__name__}: {e}")
    print(f"       Check your API key and internet connection.")

[OK] Connected to OpenAI
[OK] API key: sk-proj-...FZcA
[OK] Model 'gpt-4o' available

  Ready. No tokens used.


In [2]:
# ============================================================
# CELL 2 -- SCRAPE REDDIT POSTS
# ============================================================
# Uses the raw JSON API (no OAuth needed).
# Config at the top: subreddits, time window, limits.

import time, json, requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm.auto import tqdm

# -------- CONFIG (edit these) --------------------------------
SUBREDDITS = [
    "smallbusiness", "startups", "RestaurantOwners", "SaaS",
    "automation", "Entrepreneur", "freelance", "webdev",
    "sidehustle", "indiehackers",
]

LOOKBACK_HOURS = 24 * 7          # 7 days back from now
PAGES_PER_SUB  = 3               # 3 pages x 100 = up to 300 posts/sub
SLEEP_BETWEEN  = 2.0             # be polite to reddit
# -------------------------------------------------------------

cutoff_utc = time.time() - (LOOKBACK_HOURS * 3600)
cutoff_human = datetime.utcfromtimestamp(cutoff_utc).strftime("%Y-%m-%d %H:%M UTC")
print(f"Scraping posts newer than: {cutoff_human} ({LOOKBACK_HOURS}h ago)\n")

headers = {"User-Agent": "SaaSResearch/3.0 (academic research bot)"}
seen_ids = set()
rows = []

for sub in tqdm(SUBREDDITS, desc="Scraping"):
    after = None
    sub_n = 0

    for page in range(PAGES_PER_SUB):
        params = {"limit": 100, "raw_json": 1}
        if after:
            params["after"] = after

        try:
            r = requests.get(
                f"https://www.reddit.com/r/{sub}/new.json",
                headers=headers, params=params, timeout=15,
            )
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 15))
                tqdm.write(f"  r/{sub}: rate limited, sleeping {wait}s")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                tqdm.write(f"  r/{sub}: HTTP {r.status_code}")
                break

            data = r.json().get("data", {})
            children = data.get("children", [])
            if not children:
                break

            for child in children:
                d = child["data"]
                if d["created_utc"] < cutoff_utc:
                    continue
                if d["id"] in seen_ids:
                    continue

                seen_ids.add(d["id"])
                title = (d.get("title") or "").strip()
                body = (d.get("selftext") or "").strip()

                if not title:
                    continue

                rows.append({
                    "post_id":     d["id"],
                    "subreddit":   sub,
                    "title":       title,
                    "body":        body[:5000],
                    "url":         f"https://reddit.com{d['permalink']}",
                    "upvotes":     d.get("ups", 0),
                    "comments":    d.get("num_comments", 0),
                    "created_utc": d["created_utc"],
                    "hours_ago":   round((time.time() - d["created_utc"]) / 3600, 1),
                })
                sub_n += 1

            after = data.get("after")
            if not after:
                break

        except Exception as e:
            tqdm.write(f"  r/{sub} p{page}: {type(e).__name__}")
            break

        time.sleep(SLEEP_BETWEEN)

    tqdm.write(f"  r/{sub}: {sub_n}")

df_raw = pd.DataFrame(rows)

# -- status report --
print(f"\n{'='*55}")
print(f" SCRAPE COMPLETE")
print(f"{'='*55}")
print(f" Total posts:  {len(df_raw)}")
print(f" Unique subs:  {df_raw['subreddit'].nunique() if not df_raw.empty else 0}")
print(f" Time window:  last {LOOKBACK_HOURS}h")

if not df_raw.empty:
    print(f" Age range:    {df_raw['hours_ago'].min():.0f}h - {df_raw['hours_ago'].max():.0f}h")
    print(f" Avg upvotes:  {df_raw['upvotes'].mean():.1f}")
    print(f" Avg comments: {df_raw['comments'].mean():.1f}")
    print(f"\n Per subreddit:")
    for sub, n in df_raw["subreddit"].value_counts().items():
        print(f"   r/{sub:25s} {n:4d}")
else:
    print("\n [!] No posts found. Increase LOOKBACK_HOURS or check sub names.")

df_raw.head(3)

/opt/miniconda3/envs/cenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/yj/2dw3syk175145_7bts9_4vqr0000gn/T/ipykernel_28146/1727227543.py:25: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  cutoff_human = datetime.utcfromtimestamp(cutoff_utc).strftime("%Y-%m-%d %H:%M UTC")


Scraping posts newer than: 2026-05-02 23:42 UTC (168h ago)



Scraping:  10%|█         | 1/10 [00:09<01:24,  9.34s/it]

  r/smallbusiness: 300


Scraping:  20%|██        | 2/10 [00:19<01:20, 10.00s/it]

...

Scraping:  90%|█████████ | 9/10 [01:34<00:10, 10.62s/it]

  r/sidehustle: 21


Scraping: 100%|██████████| 10/10 [01:46<00:00, 10.69s/it]

  r/indiehackers: 30

 SCRAPE COMPLETE
 Total posts:  1208
 Unique subs:  10
 Time window:  last 168h
 Age range:    0h - 168h
 Avg upvotes:  19.8
 Avg comments: 17.8

 Per subreddit:
   r/smallbusiness              300
   r/SaaS                       300
   r/webdev                     236
   r/automation                 135
   r/startups                   115
   r/Entrepreneur                44
   r/indiehackers                30
   r/RestaurantOwners            21
   r/sidehustle                  21
   r/freelance                    6


,post_id,subreddit,title,body,url,upvotes,comments,created_utc,hours_ago
0,1t8qg3u,smallbusiness,Building a personalized moon phase bracelet br...,"Hey everyone. I am 21 years old, based in Germ...",https://reddit.com/r/smallbusiness/comments/1t...,1,1,1.778369e+09,0.2
1,1t8qcmx,smallbusiness,Was I wrong for refusing a mobile detailing jo...,I run a mobile detailing business and had a cu...,https://reddit.com/r/smallbusiness/comments/1t...,2,10,1.778369e+09,0.3
2,1t8pl5v,smallbusiness,What shirt sizes to bring to flea market,I’m selling sports shirts at a flea market and...,https://reddit.com/r/smallbusiness/comments/1t...,1,3,1.778367e+09,0.9


In [3]:
# ============================================================
# CELL 3 -- NLP KEYWORD FILTER + PAIN SCORING (local, free)
# ============================================================
# Uses keyword matching + TextBlob sentiment + engagement
# to score posts. No API calls. No tokens spent.

import numpy as np
from textblob import TextBlob

assert not df_raw.empty, "df_raw is empty, rerun cell 2"

# -------- PAIN KEYWORD LISTS (edit these) --------------------

# Stage 1: post must contain >= 1 of these to survive
GATE_KEYWORDS = [
    "how to", "how do i", "is there a", "is there any",
    "issue", "problem", "frustrated", "struggling", "annoying",
    "tool for", "looking for", "alternative to", "switch from",
    "anyone know", "any recommendations", "recommend", "suggestion",
    "manually", "waste time", "need help", "better way",
    "too expensive", "wish there was", "hate using", "sick of",
    "pain", "headache", "nightmare", "broken",
]

# Stage 2: weighted pain signals for scoring
PAIN_SIGNALS = {
    # === Buying intent (highest value) ===
    "willing to pay":       35,
    "take my money":        35,
    "looking for tool":     28,
    "looking for software": 28,
    "looking for app":      28,
    "alternative to":       25,
    "switch from":          25,
    "wish there was":       25,
    "need a tool":          25,

    # === Active frustration ===
    "hate":          14,
    "frustrated":    14,
    "annoying":      10,
    "nightmare":     16,
    "headache":      12,
    "sick of":       14,
    "fed up":        14,
    "waste time":    16,
    "too much time": 16,
    "hours every":   16,
    "tedious":       12,
    "painful":       12,
    "broken":        10,

    # === Manual process signals ===
    "manually":    12,
    "spreadsheet": 16,
    "excel":       14,
    "copy paste":  14,
    "by hand":     12,

    # === Discovery/search signals ===
    "recommendations": 14,
    "anyone know":     10,
    "does anyone":      8,
    "better way":      12,
    "any suggestions":  8,
    "need help":        6,
}

MIN_PAIN_SCORE = 10  # threshold to keep
# -------------------------------------------------------------

# Combine title + body for analysis
df_raw["full_text"] = (df_raw["title"] + " " + df_raw["body"]).str.strip()
ft_lower = df_raw["full_text"].str.lower()

# -- Stage 1: gate filter (must match >= 1 keyword) -----------
gate_mask = pd.Series(False, index=df_raw.index)
for kw in GATE_KEYWORDS:
    gate_mask |= ft_lower.str.contains(kw, regex=False)

df_gated = df_raw.loc[gate_mask].copy()
dropped_gate = len(df_raw) - len(df_gated)
print(f"Gate filter:  {len(df_raw)} --> {len(df_gated)}  ({dropped_gate} removed)")

# -- Stage 2: pain signal scoring (vectorized) ----------------
ft_gated = df_gated["full_text"].str.lower()

pain_score = pd.Series(0.0, index=df_gated.index)
matched_signals = pd.Series("", index=df_gated.index)

for signal, weight in PAIN_SIGNALS.items():
    mask = ft_gated.str.contains(signal, regex=False)
    pain_score += mask.astype(float) * weight
    matched_signals = matched_signals.where(~mask, matched_signals + signal + ", ")

df_gated["pain_score"] = pain_score

# -- Engagement score (popular pain = validated pain) ----------
df_gated["engagement_score"] = (
    np.log1p(df_gated["upvotes"]) * 2.0
    + np.log1p(df_gated["comments"]) * 3.0
).round(1)

# -- Sentiment (negative = more pain, done locally via TextBlob)
df_gated["sentiment"] = df_gated["full_text"].apply(
    lambda t: round(TextBlob(t).sentiment.polarity, 3)
)
# Negative sentiment boosts score (flip sign, clamp to 0+)
df_gated["sentiment_boost"] = df_gated["sentiment"].clip(upper=0).abs() * 15

# -- Combined rule score ---------------------------------------
df_gated["rule_score"] = (
    df_gated["pain_score"]
    + df_gated["engagement_score"]
    + df_gated["sentiment_boost"]
).round(1)

df_gated["matched_signals"] = matched_signals.str.rstrip(", ")

# -- Apply threshold -------------------------------------------
df_filtered = (
    df_gated.loc[df_gated["rule_score"] >= MIN_PAIN_SCORE]
    .sort_values("rule_score", ascending=False)
    .reset_index(drop=True)
    .copy()
)

dropped_score = len(df_gated) - len(df_filtered)
print(f"Score filter: {len(df_gated)} --> {len(df_filtered)}  ({dropped_score} below {MIN_PAIN_SCORE})")

# -- Stats -----------------------------------------------------
print(f"\n{'='*55}")
print(f" FILTERED: {len(df_filtered)} posts ready for LLM")
print(f"{'='*55}")

if not df_filtered.empty:
    print(f" Score range:    {df_filtered['rule_score'].min():.0f} -- {df_filtered['rule_score'].max():.0f}")
    print(f" Avg sentiment:  {df_filtered['sentiment'].mean():.3f} (negative = more pain)")
    print(f" Avg engagement: {df_filtered['engagement_score'].mean():.1f}")

    print(f"\n Top 10 posts:")
    for i, row in df_filtered.head(10).iterrows():
        print(f"  [{row['rule_score']:5.1f}] r/{row['subreddit']:20s} {row['title'][:65]}")
        if row["matched_signals"]:
            print(f"          signals: {row['matched_signals'][:80]}")

    print(f"\n By subreddit:")
    for sub, n in df_filtered["subreddit"].value_counts().items():
        print(f"   r/{sub:25s} {n:3d}")
else:
    print(" [!] Nothing passed. Lower MIN_PAIN_SCORE or broaden GATE_KEYWORDS.")

df_filtered.head(3)

Gate filter:  1208 --> 606  (602 removed)
Score filter: 606 --> 351  (255 below 10)

 FILTERED: 351 posts ready for LLM
 Score range:    10 -- 64
 Avg sentiment:  0.093 (negative = more pain)
 Avg engagement: 11.5

 Top 10 posts:
  [ 63.6] r/automation           Cut my monday trend research from 3 hours to 10 minutes after sti
          signals: hours every, painful, manually, spreadsheet
  [ 60.2] r/automation           We talk a lot about what we automate. What do you actively CHOOSE
          signals: waste time, manually, spreadsheet
  [ 54.6] r/indiehackers         I was building the wrong things. So I built a system to stop doin
          signals: frustrated, manually, by hand
  [ 52.6] r/webdev               any good alternative to trustpilot yet?
          signals: alternative to, hate
  [ 52.0] r/automation           Built an AI that automates the most underestimated time sink at w
          signals: hate, hours every, manually
  [ 47.3] r/automation           Anyone else wast

,post_id,subreddit,title,body,url,upvotes,comments,created_utc,hours_ago,full_text,pain_score,engagement_score,sentiment,sentiment_boost,rule_score,matched_signals
0,1t8evck,automation,Cut my monday trend research from 3 hours to 1...,Im the head of marketing at a small outdoor ge...,https://reddit.com/r/automation/comments/1t8ev...,3,4,1.778354e+09,4.5,Cut my monday trend research from 3 hours to 1...,56.0,7.6,0.089,0.0,63.6,"hours every, painful, manually, spreadsheet"
1,1t4glko,automation,We talk a lot about what we automate. What do ...,It seems like almost every post here is about ...,https://reddit.com/r/automation/comments/1t4gl...,15,34,1.777990e+09,105.6,We talk a lot about what we automate. What do ...,44.0,16.2,0.096,0.0,60.2,"waste time, manually, spreadsheet"
2,1t66z93,indiehackers,I was building the wrong things. So I built a ...,"When I started looking for my next idea, I had...",https://reddit.com/r/indiehackers/comments/1t6...,8,57,1.778151e+09,60.8,I was building the wrong things. So I built a ...,38.0,16.6,0.117,0.0,54.6,"frustrated, manually, by hand"


In [5]:
# ============================================================
# CELL 4 -- LLM ANALYSIS (only what NLP can't do)
# ============================================================
# The LLM does ONE job: extract structured insight from text.
# All scoring math happens in Python afterward.

import json, time
from tqdm.auto import tqdm

assert not df_filtered.empty, "df_filtered is empty, rerun cell 3"

# -------- CONFIG ---------------------------------------------
LLM_MAX_CHARS  = 2500   # truncate input to save tokens
LLM_SLEEP      = 0.4    # rate limit buffer
LLM_RETRIES    = 1
LLM_TEMP       = 0.1    # low temp = consistent structured output
# -------------------------------------------------------------

SYSTEM_PROMPT = """Extract structured data from this Reddit post.
Return ONLY valid JSON, no markdown fences, with exactly these keys:

{
  "pain_summary": "one sentence: what is the core problem",
  "category": "one of: workflow, cost, tooling, integration, support, data, compliance, other",
  "who_has_pain": "specific role or business type affected",
  "current_solution": "how they handle it now, or null",
  "competitors_mentioned": ["list", "of", "tools"] or [],
  "is_recurring": true or false,
  "urgency": 1-10 integer (10 = needs fix today),
  "market_size_hint": "one of: niche, medium, broad"
}

Be factual. Only extract what the post actually says or strongly implies."""


def call_llm(text):
    """Single LLM call with retry. Returns parsed dict or None."""
    for attempt in range(1 + LLM_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                temperature=LLM_TEMP,
                max_tokens=300,
                timeout=30,
            )
            parsed = json.loads(resp.choices[0].message.content)

            # validate urgency
            u = parsed.get("urgency")
            if not (isinstance(u, (int, float)) and 1 <= u <= 10):
                parsed["urgency"] = 5
            else:
                parsed["urgency"] = int(u)

            # normalize category
            valid_cats = {"workflow", "cost", "tooling", "integration", "support", "data", "compliance", "other"}
            if parsed.get("category", "").lower() not in valid_cats:
                parsed["category"] = "other"
            else:
                parsed["category"] = parsed["category"].lower()

            # normalize market_size_hint
            valid_mkt = {"niche", "medium", "broad"}
            if parsed.get("market_size_hint", "").lower() not in valid_mkt:
                parsed["market_size_hint"] = "medium"
            else:
                parsed["market_size_hint"] = parsed["market_size_hint"].lower()

            # normalize is_recurring
            parsed["is_recurring"] = bool(parsed.get("is_recurring", False))

            # normalize competitors to list
            comp = parsed.get("competitors_mentioned", [])
            if isinstance(comp, str):
                parsed["competitors_mentioned"] = [c.strip() for c in comp.split(",") if c.strip()]
            elif not isinstance(comp, list):
                parsed["competitors_mentioned"] = []

            return parsed

        except json.JSONDecodeError:
            if attempt < LLM_RETRIES:
                time.sleep(1)
                continue
            return None
        except Exception as e:
            if attempt < LLM_RETRIES:
                time.sleep(2)
                continue
            tqdm.write(f"  LLM error: {type(e).__name__}: {e}")
            return None

    return None


# -- Run LLM on each post --------------------------------------
results = []
failed = []

for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="LLM analysis"):
    # send title + truncated body (token efficient)
    input_text = f"Title: {row['title']}\n\nBody: {row['body'][:LLM_MAX_CHARS]}"
    analysis = call_llm(input_text)

    if analysis is None:
        failed.append(row["post_id"])
        continue

    results.append({
        # -- from LLM --
        "pain_summary":           analysis.get("pain_summary", ""),
        "category":               analysis["category"],
        "who_has_pain":           analysis.get("who_has_pain", ""),
        "current_solution":       analysis.get("current_solution") or "",
        "competitors_mentioned":  ", ".join(analysis.get("competitors_mentioned", [])),
        "is_recurring":           analysis["is_recurring"],
        "urgency":                analysis["urgency"],
        "market_size_hint":       analysis["market_size_hint"],
        # -- from previous cells (free, no tokens) --
        "rule_score":             row["rule_score"],
        "pain_score":             row["pain_score"],
        "engagement_score":       row["engagement_score"],
        "sentiment":              row["sentiment"],
        "matched_signals":        row["matched_signals"],
        # -- metadata --
        "subreddit":              row["subreddit"],
        "title":                  row["title"],
        "upvotes":                row["upvotes"],
        "comments":               row["comments"],
        "hours_ago":              row["hours_ago"],
        "url":                    row["url"],
    })

    time.sleep(LLM_SLEEP)

df_analyzed = pd.DataFrame(results)

# -- Compute final composite score in Python (not LLM) ---------
if not df_analyzed.empty:
    # Market size multiplier
    mkt_mult = df_analyzed["market_size_hint"].map({"niche": 0.7, "medium": 1.0, "broad": 1.3})

    # Recurring problems are more valuable
    recur_mult = df_analyzed["is_recurring"].astype(float) * 0.2 + 1.0  # 1.0 or 1.2

    # Composite: urgency (LLM) + rule_score (NLP) + engagement, scaled by market & recurrence
    raw_composite = (
        df_analyzed["urgency"]          * 4.0    # LLM signal, weighted heavy
        + df_analyzed["rule_score"]     * 0.5    # NLP keyword signal
        + df_analyzed["engagement_score"] * 0.8  # social proof
    )
    df_analyzed["final_score"] = (raw_composite * mkt_mult * recur_mult).round(1)

    df_analyzed = df_analyzed.sort_values("final_score", ascending=False).reset_index(drop=True)

print(f"\n{'='*55}")
print(f" LLM ANALYSIS COMPLETE")
print(f"{'='*55}")
print(f" Analyzed:  {len(results)}")
print(f" Failed:    {len(failed)}")

if not df_analyzed.empty:
    print(f" Score range: {df_analyzed['final_score'].min()} -- {df_analyzed['final_score'].max()}")
    print(f"\n Categories:")
    for cat, n in df_analyzed["category"].value_counts().items():
        print(f"   {cat:15s} {n:3d}")
    print(f"\n Market sizes:")
    for ms, n in df_analyzed["market_size_hint"].value_counts().items():
        print(f"   {ms:15s} {n:3d}")
    print(f" Recurring:  {df_analyzed['is_recurring'].sum()}/{len(df_analyzed)}")

if failed:
    print(f"\n Failed post IDs: {failed[:10]}")

df_analyzed.head(3)

LLM analysis: 100%|██████████| 351/351 [09:07<00:00,  1.56s/it]


 LLM ANALYSIS COMPLETE
 Analyzed:  351
 Failed:    0
 Score range: 16.3 -- 103.0

 Categories:
   workflow        209
   tooling          42
   other            33
   cost             24
   compliance       23
   integration      11
   data              6
   support           3

 Market sizes:
   medium          224
   broad            79
   niche            48
 Recurring:  291/351


,pain_summary,category,who_has_pain,current_solution,competitors_mentioned,is_recurring,urgency,market_size_hint,rule_score,pain_score,engagement_score,sentiment,matched_signals,subreddit,title,upvotes,comments,hours_ago,url,final_score
0,The core problem is that software development ...,workflow,software developers,,,True,5,broad,44.0,14.0,30.0,0.159,hate,webdev,Beign software developer doesn't make sense an...,588,309,39.5,https://reddit.com/r/webdev/comments/1t71lff/b...,103.0
1,Employees spend around 3 hours daily manually ...,workflow,employees,manual reading and responding to messages,,True,7,broad,52.0,42.0,10.0,0.067,"hate, hours every, manually",automation,Built an AI that automates the most underestim...,3,10,63.6,https://reddit.com/r/automation/comments/1t641...,96.7
2,Many business owners struggle to generate prof...,workflow,business owners,Hiring agencies or attempting to run ads thems...,Meta,True,7,broad,33.9,14.0,19.9,0.152,hate,Entrepreneur,How I'm generating 6 - 13 b2b calls per day wi...,32,74,109.8,https://reddit.com/r/Entrepreneur/comments/1t4...,95.0


In [6]:
# ============================================================
# CELL 5 -- DISPLAY + SAVE CSV
# ============================================================

from datetime import datetime

assert not df_analyzed.empty, "df_analyzed is empty, rerun cell 4"

# -- Display top results ----------------------------------------
print(f"{'='*60}")
print(f" TOP 15 SAAS OPPORTUNITIES")
print(f"{'='*60}")

for i, row in df_analyzed.head(15).iterrows():
    print(f"\n  #{i+1}  [Score: {row['final_score']}]")
    print(f"  Pain:       {row['pain_summary']}")
    print(f"  Category:   {row['category']} | Market: {row['market_size_hint']} | Recurring: {row['is_recurring']}")
    print(f"  Who:        {row['who_has_pain']}")
    print(f"  Current:    {row['current_solution']}")
    print(f"  Competitors:{row['competitors_mentioned'] or 'none found'}")
    print(f"  Urgency:    {row['urgency']}/10 | NLP pain: {row['pain_score']} | Engagement: {row['engagement_score']}")
    print(f"  r/{row['subreddit']} | ^{row['upvotes']} | {row['comments']} comments")
    print(f"  {row['url']}")

# -- Save CSV ---------------------------------------------------
csv_name = f"saas_opportunities_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_analyzed.to_csv(csv_name, index=False)

print(f"\n{'='*60}")
print(f" SAVED: {csv_name}")
print(f" ROWS:  {len(df_analyzed)}")
print(f" COLS:  {list(df_analyzed.columns)}")
print(f"{'='*60}")

df_analyzed.head(10)

 TOP 15 SAAS OPPORTUNITIES

  #1  [Score: 103.0]
  Pain:       The core problem is that software development tasks feel less meaningful and anyone can make changes without deep understanding.
  Category:   workflow | Market: broad | Recurring: True
  Who:        software developers
  Current:    
  Competitors:none found
  Urgency:    5/10 | NLP pain: 14.0 | Engagement: 30.0
  r/webdev | ^588 | 309 comments
  https://reddit.com/r/webdev/comments/1t71lff/beign_software_developer_doesnt_make_sense_anymore/

  #2  [Score: 96.7]
  Pain:       Employees spend around 3 hours daily manually reading and responding to messages.
  Category:   workflow | Market: broad | Recurring: True
  Who:        employees
  Current:    manual reading and responding to messages
  Competitors:none found
  Urgency:    7/10 | NLP pain: 42.0 | Engagement: 10.0
  r/automation | ^3 | 10 comments
  https://reddit.com/r/automation/comments/1t641g5/built_an_ai_that_automates_the_most/

  #3  [Score: 95.0]
  Pain:      

,pain_summary,category,who_has_pain,current_solution,competitors_mentioned,is_recurring,urgency,market_size_hint,rule_score,pain_score,engagement_score,sentiment,matched_signals,subreddit,title,upvotes,comments,hours_ago,url,final_score
0,The core problem is that software development ...,workflow,software developers,,,True,5,broad,44.0,14.0,30.0,0.159,hate,webdev,Beign software developer doesn't make sense an...,588,309,39.5,https://reddit.com/r/webdev/comments/1t71lff/b...,103.0
1,Employees spend around 3 hours daily manually ...,workflow,employees,manual reading and responding to messages,,True,7,broad,52.0,42.0,10.0,0.067,"hate, hours every, manually",automation,Built an AI that automates the most underestim...,3,10,63.6,https://reddit.com/r/automation/comments/1t641...,96.7
2,Many business owners struggle to generate prof...,workflow,business owners,Hiring agencies or attempting to run ads thems...,Meta,True,7,broad,33.9,14.0,19.9,0.152,hate,Entrepreneur,How I'm generating 6 - 13 b2b calls per day wi...,32,74,109.8,https://reddit.com/r/Entrepreneur/comments/1t4...,95.0
3,The user needs a free VPN for PC that doesn't ...,tooling,remote worker at an airport,,,True,8,broad,37.8,28.0,9.8,0.146,"annoying, anyone know, does anyone",automation,Best free VPN for PC with no registration and ...,4,8,128.4,https://reddit.com/r/automation/comments/1t3kr...,91.6
4,Slow website performance negatively impacts us...,workflow,web development teams,,,True,7,broad,28.8,10.0,18.8,0.108,broken,webdev,People underestimate how much trust a fast web...,100,23,38.2,https://reddit.com/r/webdev/comments/1t72w14/p...,89.6
5,"Companies are automating broken processes, lea...",workflow,companies implementing AI,Conducting a 5-step audit to clarify and fix p...,,True,7,broad,36.2,22.0,13.8,-0.027,"broken, manually",automation,AI won't fix a broken process. It'll just make...,12,17,59.7,https://reddit.com/r/automation/comments/1t68h...,89.1
6,Developer documentation is often difficult to ...,tooling,developers,"Using tools like Docusaurus, GitBook, or Notio...","Docusaurus, GitBook, Notion",True,7,broad,40.4,28.0,10.9,-0.099,"hate, frustrated",automation,Most dev docs are either hell to read or hell ...,4,12,77.2,https://reddit.com/r/automation/comments/1t5lk...,88.8
7,Communication infrastructure becomes complex a...,workflow,web app developers,,,True,7,broad,28.3,10.0,18.3,0.067,broken,webdev,I underestimated how hard communication infras...,79,23,19.2,https://reddit.com/r/webdev/comments/1t7uqwq/i...,88.6
8,Struggling to decide when to stop adding featu...,workflow,app developer,Testing with around 10 test flight users and c...,,True,5,broad,40.9,22.0,18.9,0.193,"hate, any suggestions",indiehackers,When do you stop?,12,98,50.1,https://reddit.com/r/indiehackers/comments/1t6...,86.7
9,Zapier is becoming too expensive.,cost,users of automation platforms,,"Make, n8n, wrk",True,5,broad,36.1,16.0,20.1,0.016,nightmare,automation,what are people switching to instead of Zapier?,40,67,34.0,https://reddit.com/r/automation/comments/1t78c...,84.4
